# 02 — Run Qwen3.6-35B-A3B APEX Nano (35B MoE on 15 GB VRAM!)

This is the headline act: a **35-billion-parameter Mixture-of-Experts model** running on a **free Colab T4**. Qwen3.6-35B-A3B has 35B total params but only activates ~3B per token, and the **APEX Nano** quantization (~2.5 bpw) compresses it to 11.69 GB — fitting comfortably in the T4's 15 GB VRAM.

| Metric | Value |
|--------|-------|
| File size | 11.69 GB |
| Load time | 79.8s |
| Status | ✅ Proven to fit on T4 |

## 1. Install llama-cpp-python with CUDA

In [ ]:
%%capture
pip install llama-cpp-python huggingface_hub --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124


## 2. Download Qwen3.6-35B-A3B APEX Nano GGUF

In [ ]:
from huggingface_hub import hf_hub_download
import os, time

repo_id = "mudler/Carnice-Qwen3.6-MoE-35B-A3B-APEX-MTP-GGUF"
filename = "Carnice-Qwen3.6-MoE-35B-A3B-APEX-Nano-00001-of-00003.gguf"

# APEX Nano is split across multiple files; hf_hub_download handles the shards
# Adjust filename to the exact shard name on the repo if different
t0 = time.time()
model_path = hf_hub_download(repo_id=repo_id, filename=filename)
print(f"Downloaded in {time.time()-t0:.1f}s")
print(f"Path: {model_path}")
print(f"Shard size: {os.path.getsize(model_path) / 1024**3:.2f} GB")


> **Note:** The APEX Nano quant is split into multiple GGUF shards. `hf_hub_download` fetches the named shard; `llama-cpp-python` automatically loads all shards when given the first one. If the repo has a single-file variant, use that instead.

You can list all available files in the repo with:

In [ ]:
from huggingface_hub import list_repo_files
files = list_repo_files("mudler/Carnice-Qwen3.6-MoE-35B-A3B-APEX-MTP-GGUF")
for f in files:
    print(f)


## 3. Load the model onto the GPU

In [ ]:
from llama_cpp import Llama
import time

t0 = time.time()
llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,   # all layers to GPU
    n_ctx=8192,        # large context for MoE
    verbose=False,
)
print(f"\nModel loaded in {time.time()-t0:.1f}s")


## 4. Check VRAM

In [ ]:
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv


## 5. Run inference

In [ ]:
prompt = "Write a Python function to check if a number is prime, with a docstring and type hints."

resp = llm(
    prompt,
    max_tokens=300,
    temperature=0.7,
    stop=["<|im_end|>", "<|endoftext|>"],
    verbose=False,
)
print(resp['choices'][0]['text'])


## How does a 35B model fit in 15 GB?

Three things make this possible:
1. **Mixture-of-Experts (MoE):** Only ~3B of the 35B params are active per token, so inference compute is cheap.
2. **APEX Nano quantization (~2.5 bpw):** Aggressively compresses weights to fit the full 35B in 11.69 GB.
3. **T4's 15 GB VRAM:** Just enough headroom for weights + KV cache at 8K context.

See the README for more on APEX quantization and how `ik_llama.cpp` can speed this up further.